# Tokenization and regular expressions in ToxicChat

**Assignment 1, Task 3 · Shahzaib Khan**

This notebook compares NLTK Treebank and spaCy on the same three prompts from
the ToxicChat training split (`toxicchat0124`). It also checks how often links,
HTML tags, and repeated whitespace appear in that split.

In [ ]:
from pathlib import Path
import re

import pandas as pd
import spacy
from nltk.tokenize import TreebankWordTokenizer

In [ ]:
# Reuse the original_text column preserved by the team's Assignment 1 export
# when available. On the tokenization-only branch, load the official dataset.
csv = next((root / "data/processed/toxicchat_train_cleaned.csv"
            for root in (Path.cwd(), Path.cwd().parent)
            if (root / "data/processed/toxicchat_train_cleaned.csv").exists()), None)
if csv is None:
    from datasets import load_dataset
    train_df = load_dataset("lmsys/toxic-chat", "toxicchat0124", split="train").to_pandas()
else:
    train_df = pd.read_csv(csv).rename(columns={"original_text": "user_input"})
assert len(train_df) == 5082
assert {"user_input", "toxicity"} <= set(train_df)
print("Training rows:", len(train_df))

## 1. Choose three examples

The comparison uses a short prompt with a contraction, a longer prompt, and
a prompt containing a URL. For each category, the code takes the first
eligible row labelled non-toxic. The displayed row numbers let the comparison
be reproduced without printing the full prompts.

In [ ]:
texts = train_df["user_input"].fillna("")
length = texts.str.split().str.len()
non_toxic = train_df["toxicity"].eq(0)
url_pattern = re.compile(r"""\b(?:https?://|www\.)[^\s<>"']+""", re.I)
html_pattern = re.compile(r"</?[A-Za-z][\w:-]*(?:\s+[^<>]*?)?\s*/?>")
whitespace_pattern = re.compile(r"\s{2,}")
contraction = re.compile(r"\b[A-Za-z]+n['’]t\b", re.I)
filters = {
    "short contraction": length.between(3, 14) & texts.str.contains(contraction),
    "long prompt": length.between(45, 80),
    "URL prompt": length.between(3, 40) & texts.str.contains(url_pattern),
}
examples = {}
for label, mask in filters.items():
    row = texts[non_toxic & mask & ~texts.index.isin(examples)].index[0]
    examples[row] = (label, texts.at[row])
    print(row, label, "words:", length.at[row])

## 2. Tokenize with NLTK Treebank

Treebank uses rules for punctuation and contractions. Each prompt is passed
directly to the word tokenizer, so this comparison needs no NLTK sentence
model.

In [ ]:
treebank = TreebankWordTokenizer()
nltk_tokens = {row: treebank.tokenize(text) for row, (_, text) in examples.items()}

## 3. Compare with spaCy

The blank English spaCy pipeline provides a tokenizer without downloading a
trained model. Whitespace-only tokens are excluded from its output. The table
checks both the number of tokens and whether the two token sequences are
identical. Equal counts alone would miss differences in token boundaries.

In [ ]:
nlp = spacy.blank("en")
spacy_tokens = {row: [t.text for t in nlp.make_doc(text) if not t.is_space]
                for row, (_, text) in examples.items()}
comparison = pd.DataFrame([
    {"row": row, "example": label, "NLTK tokens": len(nltk_tokens[row]),
     "spaCy tokens": len(spacy_tokens[row]), "same token sequence": nltk_tokens[row] == spacy_tokens[row]}
    for row, (label, _) in examples.items()
])
display(comparison)

## 4. Regular expressions

| Pattern | Purpose | Limitation |
|---|---|---|
| URL | Detect HTTP(S) and www links | Can include trailing punctuation and misses bare domains |
| HTML | Detect common opening, closing and self-closing tags | A heuristic that can confuse text with markup and cannot parse arbitrary HTML |
| Repeated whitespace | Find runs of two or more Unicode whitespace characters | A single newline is outside this pattern and spacing can matter in code |

Patterns detect spans without editing the source. The examples below are written
for demonstration, and the counts are calculated from the real training split.

In [ ]:
patterns = {
    "URL": (url_pattern, "See https://example.org/help now."),
    "HTML": (html_pattern, "Use <b>bold</b> here."),
    "Whitespace": (whitespace_pattern, "one  two\n\nthree"),
}
for label, (pattern, example) in patterns.items():
    matches = pattern.findall(example)
    print(label, repr(pattern.pattern), "input:", repr(example), "matches:", matches,
          "matching training rows:", int(texts.str.contains(pattern).sum()))

## 5. Look at two tricky cases

**Contractions.** Both tokenizers split the contraction in row 432 into
`Ca` and `n't`. The suffix carries the negation, so dropping it during
cleaning could change the prompt's meaning.

**URLs.** On row 699, Treebank produced five tokens for the link and spaCy
produced one. Replacing a link with a placeholder would keep a signal that
a URL was present while avoiding link-specific tokens. The cell below checks
both spans directly and prints only the URL token counts.

In [ ]:
for row, (label, text) in examples.items():
    if label == "short contraction":
        span = contraction.search(text).group()
        print("Contraction row", row, "NLTK:", treebank.tokenize(span),
              "spaCy:", [t.text for t in nlp.make_doc(span)])
    if label == "URL prompt":
        span = url_pattern.search(text).group()
        print("URL row", row, "NLTK token count:", len(treebank.tokenize(span)),
              "spaCy token count:", len(nlp.make_doc(span)))

## 6. What the comparison showed

Both methods produced nine tokens and the same sequence for the short prompt.
For the long prompt, Treebank produced 55 tokens and spaCy produced 73. The
URL prompt also differed (16 versus 12). These three examples illustrate where
token boundaries can change. They cannot tell us how often the two methods
disagree across the corpus.

Repeated whitespace was the most common of the three patterns tested, appearing
in 623 training rows. URLs appeared in 38 rows and HTML-like tags in 31. The
regexes only flag possible matches. For example, repeated spaces can carry
meaning in a code snippet, so a match alone is a poor reason to remove them.

## Reference

Lin et al. (2023), *ToxicChat: Unveiling Hidden Challenges of Toxicity Detection
in Real-World User-AI Conversation*. [Dataset card and license](https://huggingface.co/datasets/lmsys/toxic-chat).